In [1]:
# Import Python Standard Library dependencies
from copy import copy
import datetime
from glob import glob
import json
import math
import multiprocessing
import os
from pathlib import Path
import random
import urllib.request

# Import utility functions
from cjm_pandas_utils.core import markdown_to_pandas
from cjm_pil_utils.core import resize_img, get_img_files
from cjm_psl_utils.core import download_file, file_extract
from cjm_pytorch_utils.core import set_seed, pil_to_tensor, tensor_to_pil, get_torch_device, denorm_img_tensor
from cjm_torchvision_tfms.core import ResizeMax, PadSquare

import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd

# Do not truncate the contents of cells and display all rows and columns
pd.set_option('max_colwidth', None, 'display.max_rows', None, 'display.max_columns', None)

# Import PIL for image manipulation
from PIL import Image

# Import timm library
import timm

# Import PyTorch dependencies
import torch
import torch.nn as nn
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader

import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2  as transforms
from torchvision.transforms.v2 import functional as TF

from torchtnt.utils import get_module_summary
from torcheval.metrics import MulticlassAccuracy

# Import tqdm for progress bar
from tqdm.auto import tqdm

In [2]:
#Set the seed for generating random numbers in PyTorch, NumPy, and Python's random module
seed = 1234
set_seed(seed)

In [3]:
device = get_torch_device()
dtype = torch.float32
device, dtype

('cuda', torch.float32)

In [4]:
#The path for the project folder
project_dir = Path(r"C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier")

#Create the project directory if it does not already exist
project_dir.mkdir(parents=True, exist_ok=True)

#Define path to store datasets
dataset_dir = project_dir/'data'
#Create the dataset directory if it does not exist
dataset_dir.mkdir(parents=True, exist_ok=True)

#Define path to store archive files
archive_dir = dataset_dir/'Archive'
#Create the archive directory if it does not exist
archive_dir.mkdir(parents=True, exist_ok=True)

#Creating a Series with the paths and converting it to a DataFrame for display
pd.Series({
    "Project Directory:": project_dir,
    "Dataset Directory:": dataset_dir,
    "Archive Directory:": archive_dir
}).to_frame().style.hide(axis='columns')

Project Directory:,C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier
Dataset Directory:,C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier\data
Archive Directory:,C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier\data\Archive


In [5]:
#Set the name of the dataset
dataset_name = 'hagrid-classification-512p-no-gesture-150k-zip'

#Construct the HuggingFace Hub dataset name by combining the username and dataset name
hf_dataset = f'cj-mills/{dataset_name}'

#Create the path to the zip file that contains the dataset
archive_path = Path(f'{archive_dir}/{dataset_name.removesuffix("-zip")}.zip')

#Create the path to the directory where the dataset will be extracted
dataset_path = Path(f'{dataset_dir}/{dataset_name.removesuffix("-zip")}')

#Creating a Series with the dataset name and paths and converting it to a DataFrame for display
pd.Series({
    "HuggingFace Dataset:": hf_dataset,
    "Archive Path:": archive_path,
    "Dataset Path": dataset_path
}).to_frame().style.hide(axis='columns')

HuggingFace Dataset:,cj-mills/hagrid-classification-512p-no-gesture-150k-zip
Archive Path:,C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier\data\Archive\hagrid-classification-512p-no-gesture-150k.zip
Dataset Path,C:\Users\harri\OneDrive\Documents\DataScienceProjects\ImageClassifier\data\hagrid-classification-512p-no-gesture-150k
